In [21]:
import os
print("CWD:", os.getcwd())
print("Arquivos em data/processed:", os.listdir("data/processed") if os.path.exists("data/processed") else "pasta data/processed não existe")



CWD: /content
Arquivos em data/processed: pasta data/processed não existe


In [22]:
import requests, json, os, datetime
import pandas as pd

URL = "https://raw.githubusercontent.com/alura-cursos/challenge2-data-science/refs/heads/main/TelecomX_Data.json"
resp = requests.get(URL, timeout=30)
resp.raise_for_status()
raw = resp.json()

os.makedirs("data/raw", exist_ok=True)
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
with open(f"data/raw/telecomx_raw_{ts}.json", "w", encoding="utf-8") as f:
    json.dump(raw, f, ensure_ascii=False, indent=2)

df = pd.json_normalize(raw)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
date_cols = [c for c in df.columns if "date" in c or "signup" in c or "last" in c]
for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors='coerce')

os.makedirs("data/processed", exist_ok=True)
df.to_parquet("data/processed/telecomx_transformed.parquet", index=False)
df.to_csv("data/processed/telecomx_transformed.csv", index=False)
print("ETL executado — arquivos salvos em data/processed")


ETL executado — arquivos salvos em data/processed


In [23]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

if os.path.exists("data/processed/telecomx_transformed.parquet"):
    df = pd.read_parquet("data/processed/telecomx_transformed.parquet")
elif os.path.exists("data/processed/telecomx_transformed.csv"):
    df = pd.read_csv("data/processed/telecomx_transformed.csv", low_memory=False)
else:
    raise FileNotFoundError("Arquivo processado não encontrado. Rode o ETL primeiro.")

print("Shape:", df.shape)
print("Colunas:", df.columns.tolist())


Shape: (7267, 21)
Colunas: ['customerid', 'churn', 'customer.gender', 'customer.seniorcitizen', 'customer.partner', 'customer.dependents', 'customer.tenure', 'phone.phoneservice', 'phone.multiplelines', 'internet.internetservice', 'internet.onlinesecurity', 'internet.onlinebackup', 'internet.deviceprotection', 'internet.techsupport', 'internet.streamingtv', 'internet.streamingmovies', 'account.contract', 'account.paperlessbilling', 'account.paymentmethod', 'account.charges.monthly', 'account.charges.total']
